In [0]:
%sql
USE CATALOG V_Commerce;

CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
catalog = "v_commerce"
gold_schema_name = "gold"

In [0]:
from pyspark.sql import functions as F

In [0]:
tb_avaliacoes_silver = spark.table("v_commerce.silver.tb_avaliacoes")
tb_catalogo_produtos_silver = spark.table("v_commerce.silver.tb_catalogo_produtos")
tb_clickstream_silver = spark.table("v_commerce.silver.tb_clickstream")
tb_clientes_silver = spark.table("v_commerce.silver.tb_clientes")
tb_pedidos_silver = spark.table("v_commerce.silver.tb_pedidos")
tb_suporte_tickets_silver = spark.table("v_commerce.silver.tb_suporte_tickets")

In [0]:
# gold.fato_avaliacoes_pedido
# Origem: silver.tb_avaliacoes + silver.tb_pedidos + silver.tb_catalogo_produtos

avaliacoes  = spark.table("v_commerce.silver.tb_avaliacoes")
pedidos     = spark.table("v_commerce.silver.tb_pedidos")
catalogo    = spark.table("v_commerce.silver.tb_catalogo_produtos") \
                   .select("id_produto", "nome_produto", "categoria", "preco")

# ── pct_recomendacoes_sim por produto ─────────────────────────────────────────
# Calcula, para cada id_produto, o percentual de avaliações com recomenda = "Sim"
pct_recomendacoes = (
    avaliacoes
    .groupBy("id_produto")
    .agg(
        F.round(
            F.sum(F.when(F.col("recomenda") == "Sim", 1).otherwise(0)) /
            F.count("id_avaliacao") * 100, 2
        ).alias("pct_recomendacoes_sim")
    )
)

fato_avaliacoes_pedido = (
    avaliacoes
    .join(pedidos.select(
        "id_pedido", "valor_pedido", "data_pedido",
        "metodo_pagamento", "status", "quantidade"
    ), on="id_pedido", how="left")

    .join(catalogo, on="id_produto", how="left")
    .join(pct_recomendacoes, on="id_produto", how="left")

    .select(
        # chaves
        F.col("id_avaliacao"),
        F.col("id_pedido"),
        F.col("id_cliente"),
        F.col("id_produto"),

        # dimensões do produto
        F.col("nome_produto"),
        F.col("categoria"),
        F.col("preco"),

        # dimensões do pedido
        F.col("valor_pedido"),
        F.col("quantidade"),
        F.col("metodo_pagamento"),
        F.col("status"),
        F.col("data_pedido"),

        # métricas de avaliação
        F.col("nota_produto"),
        F.col("nota_nps"),
        F.col("recomenda"),
        F.col("comentario"),
        F.col("data_avaliacao"),

        # categoria NPS derivada da nota_nps
        F.when(F.col("nota_nps").between(9, 10), "Promotor")
         .when(F.col("nota_nps").between(7,  8), "Neutro")
         .when(F.col("nota_nps").between(0,  6), "Detrator")
         .otherwise(None)
         .alias("categoria_nps"),

        # percentual de recomendações "Sim" para o produto
        F.col("pct_recomendacoes_sim"),

        F.current_timestamp().alias("timestamp_ingestion_gold"),
    )
)

fato_avaliacoes_pedido.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.fato_avaliacoes_pedido")

print(f"✅ Tabela {catalog}.{gold_schema_name}.fato_avaliacoes_pedido criada com sucesso!")
print(f"   Total de linhas: {fato_avaliacoes_pedido.count()}")

✅ Tabela v_commerce.gold.fato_avaliacoes_pedido criada com sucesso!
   Total de linhas: 156832


In [0]:
# gold.dim_produto
# Origem: silver.tb_catalogo_produtos + silver.tb_pedidos
#         + silver.tb_suporte_tickets + silver.tb_avaliacoes

catalogo  = spark.table("v_commerce.silver.tb_catalogo_produtos")
pedidos   = spark.table("v_commerce.silver.tb_pedidos")
tickets   = spark.table("v_commerce.silver.tb_suporte_tickets")
avaliacoes = spark.table("v_commerce.silver.tb_avaliacoes")

# ── Métricas de vendas por produto ────────────────────────────────────────────
metricas_pedidos = (
    pedidos
    .groupBy("id_produto")
    .agg(
        F.count("id_pedido")                    .alias("total_pedidos"),
        F.round(F.sum("valor_pedido"), 2)        .alias("receita_total"),
        F.round(F.avg("valor_pedido"), 2)        .alias("ticket_medio"),
        F.sum("quantidade")                      .alias("total_unidades_vendidas"),
    )
)

# ── Métricas de avaliação por produto ─────────────────────────────────────────
metricas_avaliacoes = (
    avaliacoes
    .groupBy("id_produto")
    .agg(
        F.count("id_avaliacao")                                              .alias("total_avaliacoes"),
        F.round(F.avg("nota_produto"), 2)                                    .alias("media_nota_produto"),
        F.round(F.avg("nota_nps"), 2)                                        .alias("media_nota_nps"),
        F.round(
            F.sum(F.when(F.col("recomenda") == "Sim", 1).otherwise(0)) /
            F.count("id_avaliacao") * 100, 2
        )                                                                     .alias("pct_recomendacoes_sim"),
    )
)

# ── Métricas de suporte por produto ───────────────────────────────────────────
# tickets não tem id_produto direto — liga via id_pedido
metricas_tickets = (
    tickets
    .join(pedidos.select("id_pedido", "id_produto"), on="id_pedido", how="left")
    .groupBy("id_produto")
    .agg(
        F.count("id_ticket")                          .alias("total_tickets"),
        F.round(F.avg("tempo_resolucao_horas"), 2)    .alias("media_tempo_resolucao_horas"),
        F.round(F.avg("nota_avaliacao"), 2)            .alias("media_nota_suporte"),
    )
)

# ── Montagem final da dim_produto ─────────────────────────────────────────────
dim_produto = (
    catalogo
    .join(metricas_pedidos,    on="id_produto", how="left")
    .join(metricas_avaliacoes, on="id_produto", how="left")
    .join(metricas_tickets,    on="id_produto", how="left")

    .select(
        # identificação
        F.col("id_produto"),
        F.col("nome_produto"),
        F.col("categoria"),
        F.col("fornecedor"),

        # preço e estoque
        F.col("preco"),
        F.col("peso_kg"),
        F.col("estoque_disponivel"),
        F.col("ativo"),
        F.col("avaliacao_interna"),

        # flags de qualidade
        F.col("precisa_revisao"),

        # datas
        F.col("data_cadastro_produto"),

        # métricas de vendas
        F.coalesce(F.col("total_pedidos"),          F.lit(0)).alias("total_pedidos"),
        F.coalesce(F.col("receita_total"),           F.lit(0)).alias("receita_total"),
        F.coalesce(F.col("ticket_medio"),            F.lit(0)).alias("ticket_medio"),
        F.coalesce(F.col("total_unidades_vendidas"), F.lit(0)).alias("total_unidades_vendidas"),

        # métricas de avaliação
        F.coalesce(F.col("total_avaliacoes"),        F.lit(0))   .alias("total_avaliacoes"),
        F.col("media_nota_produto"),
        F.col("media_nota_nps"),
        F.col("pct_recomendacoes_sim"),

        # métricas de suporte
        F.coalesce(F.col("total_tickets"),           F.lit(0))   .alias("total_tickets"),
        F.col("media_tempo_resolucao_horas"),
        F.col("media_nota_suporte"),

        F.current_timestamp().alias("timestamp_ingestion_gold"),
    )
)

dim_produto.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema_name}.dim_produto")

print(f"✅ Tabela {catalog}.{gold_schema_name}.dim_produto criada com sucesso!")
print(f"   Total de linhas: {dim_produto.count()}")

✅ Tabela v_commerce.gold.dim_produto criada com sucesso!
   Total de linhas: 517


In [0]:
# gold.dim_tempo
# Gerada sinteticamente a partir do intervalo de datas do dataset
# Fonte de referência: tb_pedidos_silver (data_pedido)

pedidos = spark.table("v_commerce.silver.tb_pedidos")

# ── Extrai intervalo de datas do dataset ──────────────────────────────────────
datas = pedidos.agg(
    F.min("data_pedido").alias("data_min"),
    F.max("data_pedido").alias("data_max")
).collect()[0]

data_min = datas["data_min"]
data_max = datas["data_max"]

print(f"📅 Intervalo de datas: {data_min} → {data_max}")

# ── Gera sequência de datas entre min e max ───────────────────────────────────
dim_tempo = (
    spark.sql(f"""
        SELECT explode(sequence(
            date '{data_min}',
            date '{data_max}',
            interval 1 day
        )) AS data
    """)

    .select(
        # chave
        F.col("data"),

        # extrações diretas
        F.year("data")                          .alias("ano"),
        F.month("data")                         .alias("mes"),
        F.dayofmonth("data")                    .alias("dia"),
        F.quarter("data")                       .alias("trimestre"),
        F.dayofweek("data")                     .alias("dia_semana_num"),
        F.dayofyear("data")                     .alias("dia_do_ano"),
        F.weekofyear("data")                    .alias("semana_do_ano"),

        # descrições
        F.date_format("data", "MMMM")           .alias("nome_mes"),
        F.date_format("data", "EEEE")           .alias("nome_dia_semana"),
        F.date_format("data", "yyyy-MM")        .alias("ano_mes"),
        F.concat(F.lit("Q"), F.quarter("data")) .alias("trimestre_label"),

        # flags úteis
        F.when(F.dayofweek("data").isin(1, 7), "Sim")
         .otherwise("Nao")
         .alias("fim_de_semana"),

        F.current_timestamp()                   .alias("timestamp_ingestion_gold"),
    )
)

dim_tempo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{gold_schema_name}.dim_tempo")

print(f"✅ Tabela {catalog}.{gold_schema_name}.dim_tempo criada com sucesso!")
print(f"   Total de dias gerados: {dim_tempo.count()}")

📅 Intervalo de datas: 2023-01-01 → 2026-05-31
✅ Tabela v_commerce.gold.dim_tempo criada com sucesso!
   Total de dias gerados: 1247
